In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

RES_PATH = "development_042_silver_sandbox.demand_forecast"

# Output from Notebook 1, the PDF parser
SOURCE_TABLE = f"{RES_PATH}.pdf_pages_for_maverick_v1"

# Notebook 2's raw AI extraction output
OUTPUT_TABLE = f"{RES_PATH}.overview_ctc_line_reel_theme_raw_v2"

# Use 10 for the sample run; use -1 to process all eligible games
RUN_LIMIT = -1

In [0]:
if not spark.catalog.tableExists(SOURCE_TABLE):
    raise ValueError(
        f"Source table not found: {SOURCE_TABLE}. "
        "Set SOURCE_TABLE in Cell 1 to the Databricks table made from 02__extract_ai.csv."
    )

source_df = spark.table(SOURCE_TABLE)

required_columns = {
    "game_name",
    "file_name",
    "page_index",
    "page_number",
    "page_image_path",
    "page_text",
    "page_elements_json",
}

missing_columns = required_columns - set(source_df.columns)

if missing_columns:
    raise ValueError(
        f"Source table is missing expected columns: {sorted(missing_columns)}. "
        f"Available columns: {source_df.columns}"
    )

print(f"Source table: {SOURCE_TABLE}")
print(f"Columns: {source_df.columns}")

display(
    source_df
    .select(
        "game_name",
        "file_name",
        "page_number",
        F.length("page_text").alias("page_text_char_count"),
        F.length("page_elements_json").alias("page_elements_char_count"),
        "page_image_path",
    )
    .limit(10)
)

In [0]:
if RUN_LIMIT != -1 and RUN_LIMIT < 1:
    raise ValueError("RUN_LIMIT must be -1 or a positive integer.")

document_keys = ["game_name", "file_name", "source_path", "image_output_folder"]

document_keys_df = source_df.select(*document_keys).distinct()

if RUN_LIMIT == -1:
    source_for_run_df = source_df
else:
    selected_documents_df = (
        document_keys_df
        .orderBy("game_name", "file_name")
        .limit(RUN_LIMIT)
    )

    source_for_run_df = source_df.join(
        selected_documents_df,
        on=document_keys,
        how="inner",
    )

page_record = F.struct(
    F.col("page_index").cast("int").alias("page_index"),
    F.col("page_number").cast("int").alias("page_number"),
    F.coalesce(F.col("page_text"), F.lit("")).alias("page_text"),
    F.coalesce(F.col("page_elements_json"), F.lit("[]")).alias("page_elements_json"),
    F.col("page_image_path").alias("page_image_path"),
)

game_docs_df = (
    source_for_run_df
    .groupBy(*document_keys)
    .agg(
        F.sort_array(F.collect_list(page_record)).alias("pages")
    )
    .withColumn("page_count", F.size("pages"))
)

print(f"RUN_LIMIT: {RUN_LIMIT}")
print(f"Documents selected: {game_docs_df.count()}")

display(
    game_docs_df
    .select("game_name", "file_name", "page_count")
    .orderBy("game_name", "file_name")
)

In [0]:
game_docs_df = (
    game_docs_df
    .withColumn(
        "document_text",
        F.concat_ws(
            "\n\n",
            F.transform(
                "pages",
                lambda page: F.concat(
                    F.lit("PAGE "),
                    page["page_number"].cast("string"),
                    F.lit("\n"),
                    page["page_text"],
                    F.lit("\nPARSED PAGE ELEMENTS:\n"),
                    page["page_elements_json"],
                ),
            ),
        ),
    )
    .withColumn(
        "image_paths",
        F.filter(
            F.transform("pages", lambda page: page["page_image_path"]),
            lambda path: path.isNotNull() & (path != ""),
        ),
    )
    .withColumn("document_text_char_count", F.length("document_text"))
    .withColumn("image_page_count", F.size("image_paths"))
)

display(
    game_docs_df
    .select(
        "game_name",
        "file_name",
        "page_count",
        "document_text_char_count",
        "image_page_count",
    )
    .orderBy("game_name", "file_name")
)

In [0]:
preview_df = (
    game_docs_df
    .orderBy("game_name", "file_name")
    .limit(1)
    .withColumn(
        "document_text_preview",
        F.substring("document_text", 1, 6000),
    )
    .withColumn(
        "image_paths_preview",
        F.slice("image_paths", 1, 5),
    )
)

display(
    preview_df.select(
        "game_name",
        "file_name",
        "page_count",
        "document_text_char_count",
        "document_text_preview",
        "image_page_count",
        "image_paths_preview",
    )
)

In [0]:
# One image folder per PDF. Collect only the folders selected by RUN_LIMIT.
image_folders = [
    row["image_output_folder"]
    for row in (
        source_for_run_df
        .select("image_output_folder")
        .where(F.col("image_output_folder").isNotNull())
        .distinct()
        .collect()
    )
]

if not image_folders:
    raise ValueError(
        "No image_output_folder values found for the selected documents."
    )

# Keep page text and evidence fields alongside each image.
page_image_metadata_df = (
    source_for_run_df
    .select(
        "game_name",
        "file_name",
        "page_index",
        "page_number",
        "page_text",
        "page_elements_json",
        "page_image_path",
    )
    .where(F.col("page_image_path").isNotNull())
    .where(F.lower("page_image_path").rlike(r"\.(png|jpe?g)$"))
    .dropDuplicates(["game_name", "file_name", "page_index", "page_image_path"])
)

# Load image bytes from the selected per-game folders.
binary_images_raw_df = (
    spark.read
    .format("binaryFile")
    .load(image_folders)
)

# Normalize file URI prefixes so paths can be matched to the CSV paths.
def normalize_image_path(column):
    return F.regexp_replace(
        column,
        r"^(file:|dbfs:)/+",
        "/",
    )

binary_images_df = (
    binary_images_raw_df
    .select(
        normalize_image_path(F.col("path")).alias("image_path_key"),
        F.col("content"),
        F.col("length").alias("image_bytes"),
    )
)

page_images_df = (
    page_image_metadata_df
    .withColumn(
        "image_path_key",
        normalize_image_path(F.col("page_image_path")),
    )
    .join(binary_images_df, on="image_path_key", how="left")
)

missing_image_count = (
    page_images_df
    .where(F.col("content").isNull())
    .count()
)

available_image_count = (
    page_images_df
    .where(F.col("content").isNotNull())
    .count()
)

print(f"Page images matched: {available_image_count}")
print(f"Page image paths without a matching file: {missing_image_count}")

display(
    page_images_df
    .select(
        "game_name",
        "file_name",
        "page_number",
        "page_image_path",
        "image_bytes",
        F.col("content").isNotNull().alias("image_loaded"),
    )
    .orderBy("game_name", "file_name", "page_number")
)

In [0]:
page_elements_schema = ArrayType(
    StructType([
        StructField("type", StringType(), True),
        StructField("content", StringType(), True),
    ])
)

page_candidates_df = (
    page_image_metadata_df
    .withColumn(
        "_elements",
        F.from_json(
            F.coalesce(F.col("page_elements_json"), F.lit("[]")),
            page_elements_schema,
        ),
    )
    .withColumn(
        "figure_count",
        F.size(
            F.filter(
                F.col("_elements"),
                lambda element: F.lower(
                    F.coalesce(element["type"], F.lit(""))
                ) == "figure",
            )
        ),
    )
)

visual_page_metadata_df = (
    page_candidates_df
    .where(
        (F.col("page_number") == 1)
        | (F.col("figure_count") > 0)
    )
    .withColumn(
        "image_path_key",
        normalize_image_path(F.col("page_image_path")),
    )
    .drop("_elements")
)

visual_pages_df = (
    visual_page_metadata_df
    .join(binary_images_df, on="image_path_key", how="left")
    .where(F.col("content").isNotNull())
)

print(f"Selected image pages / potential visual calls: {visual_pages_df.count()}")

display(
    visual_pages_df
    .select(
        "game_name",
        "file_name",
        "page_number",
        "figure_count",
        "image_bytes",
        F.substring("page_text", 1, 500).alias("page_text_preview"),
    )
    .orderBy("game_name", "file_name", "page_number")
)

In [0]:
from uuid import uuid4

AI_ENDPOINT = "system.ai.llama-4-maverick"
VISION_STAGE_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_visual_stage_v2"
)
VISION_RUN_ID = uuid4().hex

VISUAL_ANALYSIS_PROMPT = """
Analyze the supplied PDF page image for evidence relevant to the game.

Read legible text embedded in the image, including feature descriptions,
mechanic rules, instructions, labels, and configuration details. Preserve a
short exact quote for any text that supports an observation. Do not rely only
on page_text from the parser; the image may contain text the parser missed.

Do not use the game name or title as evidence. Ignore logos, wordmarks, and
title text when identifying themes or mechanics. Use non-title text, artwork,
symbols, game screens, and diagrams.

Look for:
- Themes, with concise names that can be unified across games.
- Mechanics stated in embedded text or clearly illustrated.
- Perceived-persistence cues, such as a filling meter, collection build-up,
  growing feature, or visible state that appears to build over time.
- Actual persistence only when embedded text explicitly says a feature or state
  carries over between spins or rounds.
- Reel layout, symbol examples, and other configuration details when clear.

Return one valid json object only. Do not use markdown fences.:
{
  "visual_observations": [
    {
      "category": "theme | mechanic | perceived_persistence | actual_persistence | reel_or_symbol | other",
      "name": "short, consistent label",
      "description": "what the page shows or says",
      "evidence_type": "embedded_text | visual",
      "evidence": "exact short quote for embedded text, or a concise description of visible evidence"
    }
  ],
  "notes": []
}

For text that is blurry or unreadable, do not guess. If the page has no relevant
evidence, return an empty visual_observations array.
"""

visual_ai_input_df = (
    visual_pages_df
    .withColumn(
        "visual_prompt",
        F.concat(
            F.lit(VISUAL_ANALYSIS_PROMPT),
            F.lit("\nPage number: "),
            F.col("page_number").cast("string"),
        ),
    )
)

visual_ai_results_df = visual_ai_input_df.withColumn(
    "visual_ai_result",
    F.expr(
        """
        ai_query(
            'system.ai.llama-4-maverick',
            concat('Return one valid json object only. ', visual_prompt),
            files => content,
            failOnError => false
        )
        """
    ),
)
(
    visual_ai_results_df
    .select(
        "game_name",
        "file_name",
        "page_number",
        "page_image_path",
        "visual_ai_result",
    )
    .withColumn("run_id", F.lit(VISION_RUN_ID))
    .withColumn("saved_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(VISION_STAGE_TABLE)
)

print(f"Saved this run to: {VISION_STAGE_TABLE}")
print(f"Run ID: {VISION_RUN_ID}")

display(
    spark.table(VISION_STAGE_TABLE)
    .where(F.col("run_id") == VISION_RUN_ID)
    .select(
        "game_name",
        "file_name",
        "page_number",
        "visual_ai_result",
    )
    .orderBy("game_name", "file_name", "page_number")
)

In [0]:
visual_observation_schema = ArrayType(
    StructType([
        StructField("category", StringType(), True),
        StructField("name", StringType(), True),
        StructField("description", StringType(), True),
        StructField("evidence_type", StringType(), True),
        StructField("evidence", StringType(), True),
    ])
)

visual_response_schema = StructType([
    StructField("visual_observations", visual_observation_schema, True),
    StructField("notes", ArrayType(StringType()), True),
])

visual_stage_for_run_df = (
    spark.table(VISION_STAGE_TABLE)
    .where(F.col("run_id") == VISION_RUN_ID)
)

visual_rows_df = (
    visual_stage_for_run_df
    .withColumn("_result_json", F.to_json("visual_ai_result"))
    .withColumn(
        "_response_text",
        F.coalesce(
            F.get_json_object("_result_json", "$.result"),
            F.get_json_object("_result_json", "$.response"),
        ),
    )
    .withColumn(
        "_error_message",
        F.get_json_object("_result_json", "$.errorMessage"),
    )
    .withColumn(
        "_response_clean",
        F.regexp_replace(
            F.regexp_replace(
                F.col("_response_text"),
                r"(?is)^\s*```(?:json)?\s*",
                "",
            ),
            r"\s*```\s*$",
            "",
        ),
    )
    .withColumn(
        "_parsed_response",
        F.from_json("_response_clean", visual_response_schema),
    )
)

visual_observations_by_game_df = (
    visual_rows_df
    .select(
        "game_name",
        "file_name",
        "page_number",
        F.explode_outer(
            F.col("_parsed_response.visual_observations")
        ).alias("observation"),
    )
    .where(F.col("observation").isNotNull())
    .groupBy("game_name", "file_name")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("page_number").alias("page_number"),
                    F.col("observation.category").alias("category"),
                    F.col("observation.name").alias("name"),
                    F.col("observation.description").alias("description"),
                    F.col("observation.evidence_type").alias("evidence_type"),
                    F.col("observation.evidence").alias("evidence"),
                )
            )
        ).alias("visual_observations")
    )
    .withColumn(
        "visual_observations_json",
        F.to_json("visual_observations"),
    )
    .withColumn(
        "visual_observation_count",
        F.size("visual_observations"),
    )
)

visual_errors_by_game_df = (
    visual_rows_df
    .where(F.col("_error_message").isNotNull())
    .groupBy("game_name", "file_name")
    .agg(
        F.collect_list(
            F.struct(
                F.col("page_number").alias("page_number"),
                F.col("_error_message").alias("error_message"),
            )
        ).alias("visual_errors")
    )
    .withColumn("visual_errors_json", F.to_json("visual_errors"))
)

game_extraction_input_df = (
    game_docs_df
    .join(
        visual_observations_by_game_df.select(
            "game_name",
            "file_name",
            "visual_observations_json",
            "visual_observation_count",
        ),
        on=["game_name", "file_name"],
        how="left",
    )
    .join(
        visual_errors_by_game_df.select(
            "game_name",
            "file_name",
            "visual_errors_json",
        ),
        on=["game_name", "file_name"],
        how="left",
    )
    .withColumn(
        "visual_observations_json",
        F.coalesce(F.col("visual_observations_json"), F.lit("[]")),
    )
    .withColumn(
        "visual_observation_count",
        F.coalesce(F.col("visual_observation_count"), F.lit(0)),
    )
    .withColumn(
        "visual_errors_json",
        F.coalesce(F.col("visual_errors_json"), F.lit("[]")),
    )
)

print(
    "Visual AI rows with errors: "
    f"{visual_rows_df.where(F.col('_error_message').isNotNull()).count()}"
)
print(
    "Game documents ready for final extraction: "
    f"{game_extraction_input_df.count()}"
)

display(
    game_extraction_input_df
    .select(
        "game_name",
        "file_name",
        "page_count",
        "document_text_char_count",
        "visual_observation_count",
        "visual_errors_json",
        F.substring("visual_observations_json", 1, 3000)
         .alias("visual_observations_preview"),
    )
    .orderBy("game_name", "file_name")
)

In [0]:
FINAL_EXTRACTION_INSTRUCTIONS = """
You are extracting structured facts about one slot game from a PDF.

The input will contain:
1. DOCUMENT_TEXT: parsed text from the PDF, with page numbers.
2. VISUAL_OBSERVATIONS_JSON: observations from selected page images, including
   page numbers, embedded text quotes, and visual descriptions.

Use both inputs as evidence. The game name and PDF filename are metadata only.
Never infer themes or mechanics from the game name, title, logos, or wordmarks.

Return exactly one valid json object. Do not include markdown fences or prose.

General rules:
- Do not guess. Use null for an unknown scalar and [] for an unsupported list.
- Cite evidence for every extracted fact. Each evidence item needs a page number,
  source type, and a short exact quote or concise visual description.
- Use source_type values: pdf_text, image_embedded_text, or image_visual.
- Preserve the original currency and denomination for every bet and cost value.
- Keep theme and mechanic names short and as consistent across games as possible.
  Notebook 3 will perform the final taxonomy unification.
- Distinguish paylines from ways/multiway. Use "unknown" if the PDF does not
  establish the format.
- Report reel configuration, number of reels, symbol count, and line/ways count
  only when supported. Do not count decorative art as game symbols.
- Keep each stated denomination configuration separate.
- When bet limits vary by denomination, set overall_minimum_bet from the lowest
  denomination configuration and overall_maximum_bet from the highest
  denomination configuration, when those configurations are clear. Preserve
  the currency and denomination with each amount. If currencies make that
  comparison invalid, leave the overall value null and report the configurations.
- Report cost to cover only when explicitly stated or calculable from values
  provided in the PDF. Do not assume missing inputs.

Themes:
- Return every supported theme; a game can have multiple themes.
- Score each theme from 0 to 5: 0 means no presence, 1 incidental, 2 minor,
  3 clear, 4 strong, and 5 super-dominant.
- Set theme_presence_score to the highest theme score, or 0 when no supported
  theme is present.
- Themes can be supported by non-title text or visual content.

Mechanics:
- Return only mechanics supported by evidence.
- Give each returned mechanic a binary score of 1. Notebook 3 will create the
  shared mechanic columns and assign 0 when a mechanic is absent from a game's
  extracted list.
- Include a concise definition and evidence.

Actual persistence:
- Actual persistence means a game state, symbol, meter, bonus state, or other
  feature carries over between spins or rounds.
- Set score to 1 when evidence supports carryover.
- Set score to 0 only when the document explicitly says the feature does not
  persist.
- Set score to null and status to "unknown" if the document does not establish
  whether it persists. Do not treat missing evidence as proof of absence.

Perceived persistence:
- This measures cues that make progress feel ongoing to the player, even if
  actual carryover is not established. Examples include a filling meter,
  collection build-up, a growing feature, or a visible state that builds over time.
- Score from 0 to 5 using the same presence scale as themes.
- Use null only when the relevant source pages are missing, unreadable, or
  otherwise insufficient to assess. Do not infer from the title.

Use this exact object structure:
{
  "game_format": {
    "value": "paylines | ways | other | unknown",
    "evidence": []
  },
  "reel_configuration": {
    "value": null,
    "evidence": []
  },
  "number_of_reels": {
    "value": null,
    "evidence": []
  },
  "symbol_count": {
    "value": null,
    "evidence": []
  },
  "paylines_or_ways_count": {
    "value": null,
    "evidence": []
  },
  "denomination_configurations": [
    {
      "denomination": {
        "value": null,
        "currency": null,
        "evidence": []
      },
      "minimum_bet": {
        "value": null,
        "currency": null,
        "evidence": []
      },
      "maximum_bet": {
        "value": null,
        "currency": null,
        "evidence": []
      },
      "cost_to_cover": {
        "value": null,
        "currency": null,
        "evidence": []
      }
    }
  ],
  "overall_minimum_bet": {
    "value": null,
    "currency": null,
    "denomination": null,
    "evidence": []
  },
  "overall_maximum_bet": {
    "value": null,
    "currency": null,
    "denomination": null,
    "evidence": []
  },
  "mechanics": [
    {
      "name": null,
      "definition": null,
      "score": 1,
      "evidence": []
    }
  ],
  "themes": [
    {
      "name": null,
      "definition": null,
      "score": 1,
      "evidence": []
    }
  ],
  "theme_presence_score": 0,
  "persistence": {
    "status": "present | explicitly_absent | unknown",
    "score": null,
    "definition": null,
    "evidence": []
  },
  "perceived_persistence": {
    "score": 0,
    "definition": null,
    "evidence": []
  },
  "extraction_notes": []
}

Each evidence item must use this structure:
{
  "page_number": null,
  "source_type": "pdf_text | image_embedded_text | image_visual",
  "excerpt_or_visual_description": null
}

For unknown numeric or text facts, keep the value null and evidence empty.
If no supported themes or mechanics are present, return empty arrays.
"""

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

FINAL_STAGE_TABLE = f"{RES_PATH}.overview_ctc_line_reel_theme_final_stage_v1"
FINAL_RUN_ID = uuid4().hex

# Combine the Cell 11 instructions with the text and visual evidence prepared in Cell 10.
final_ai_input_df = game_extraction_input_df.withColumn(
    "final_prompt",
    F.concat_ws(
        "\n\n",
        F.lit(FINAL_EXTRACTION_INSTRUCTIONS),
        F.lit("PDF page text:"),
        F.coalesce(F.col("document_text"), F.lit("")),
        F.lit("Visual observations from the PDF pages:"),
        F.coalesce(F.col("visual_observations_json"), F.lit("[]")),
    ),
)

# One final extraction call per row in game_extraction_input_df.
# The prompt requests a JSON object; responseFormat is omitted.
final_ai_results_df = final_ai_input_df.withColumn(
    "final_ai_result",
    F.expr(
        """
        ai_query(
            'system.ai.llama-4-maverick',
            final_prompt,
            failOnError => false
        )
        """
    ),
)

# Save the raw results so the next cell can parse and validate them.
(
    final_ai_results_df
    .withColumn("run_id", F.lit(FINAL_RUN_ID))
    .withColumn("saved_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(FINAL_STAGE_TABLE)
)

print(f"Saved this run to: {FINAL_STAGE_TABLE}")
print(f"Run ID: {FINAL_RUN_ID}")

display(
    spark.table(FINAL_STAGE_TABLE)
    .where(F.col("run_id") == FINAL_RUN_ID)
    .select("game_name", "final_ai_result")
    .orderBy("game_name")
)

In [0]:
from pyspark.sql import functions as F

FINAL_PARSED_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_final_parsed_v1"
)

final_stage_for_run_df = (
    spark.table(FINAL_STAGE_TABLE)
    .where(F.col("run_id") == FINAL_RUN_ID)
)

# ai_query returns a result/error structure. Keep the response text and error
# separate so failed calls are easy to identify.
final_parsed_df = (
    final_stage_for_run_df
    .withColumn("_ai_result_json", F.to_json(F.col("final_ai_result")))
    .withColumn(
        "response_text",
        F.coalesce(
            F.get_json_object("_ai_result_json", "$.result"),
            F.get_json_object("_ai_result_json", "$.response"),
        ),
    )
    .withColumn(
        "ai_error_message",
        F.get_json_object("_ai_result_json", "$.errorMessage"),
    )
    .withColumn(
        "response_clean",
        F.trim(
            F.regexp_replace(
                F.col("response_text"),
                r"(?is)^\s*```(?:json)?\s*|\s*```\s*$",
                "",
            )
        ),
    )
    .withColumn(
        "_json_root",
        F.get_json_object(F.col("response_clean"), "$"),
    )
    .withColumn(
        "json_valid",
        F.col("_json_root").isNotNull()
        & F.col("response_clean").startswith("{")
        & F.col("response_clean").endswith("}"),
    )
    .withColumn(
        "parse_status",
        F.when(F.col("ai_error_message").isNotNull(), F.lit("ai_error"))
         .when(F.col("json_valid"), F.lit("valid_json"))
         .otherwise(F.lit("invalid_or_empty_json")),
    )
    .drop("_ai_result_json", "_json_root")
)

(
    final_parsed_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(FINAL_PARSED_TABLE)
)

print(f"Saved this run to: {FINAL_PARSED_TABLE}")

display(
    spark.table(FINAL_PARSED_TABLE)
    .where(F.col("run_id") == FINAL_RUN_ID)
    .select(
        "game_name",
        "parse_status",
        "ai_error_message",
        "response_clean",
    )
    .orderBy("game_name")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

FINAL_EXTRACTED_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_extracted_v1"
)

# Each evidence item keeps its source page and exact quote or visual description.
evidence_schema = ArrayType(
    StructType([
        StructField("page_number", IntegerType(), True),
        StructField("source_type", StringType(), True),
        StructField("excerpt_or_visual_description", StringType(), True),
    ]),
    True,
)

# Flexible value fields preserve numeric and formatted text values from the model.
value_with_evidence_schema = StructType([
    StructField("value", StringType(), True),
    StructField("evidence", evidence_schema, True),
])

bet_value_schema = StructType([
    StructField("value", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("evidence", evidence_schema, True),
])

overall_bet_schema = StructType([
    StructField("value", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("denomination", StringType(), True),
    StructField("evidence", evidence_schema, True),
])

denomination_configuration_schema = StructType([
    StructField("denomination", bet_value_schema, True),
    StructField("minimum_bet", bet_value_schema, True),
    StructField("maximum_bet", bet_value_schema, True),
    StructField("cost_to_cover", bet_value_schema, True),
])

mechanic_or_theme_schema = StructType([
    StructField("name", StringType(), True),
    StructField("definition", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("evidence", evidence_schema, True),
])

persistence_schema = StructType([
    StructField("status", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("definition", StringType(), True),
    StructField("evidence", evidence_schema, True),
])

perceived_persistence_schema = StructType([
    StructField("score", IntegerType(), True),
    StructField("definition", StringType(), True),
    StructField("evidence", evidence_schema, True),
])

final_extraction_schema = StructType([
    StructField("game_format", value_with_evidence_schema, True),
    StructField("reel_configuration", value_with_evidence_schema, True),
    StructField("number_of_reels", value_with_evidence_schema, True),
    StructField("symbol_count", value_with_evidence_schema, True),
    StructField("paylines_or_ways_count", value_with_evidence_schema, True),
    StructField(
        "denomination_configurations",
        ArrayType(denomination_configuration_schema, True),
        True,
    ),
    StructField("overall_minimum_bet", overall_bet_schema, True),
    StructField("overall_maximum_bet", overall_bet_schema, True),
    StructField(
        "mechanics",
        ArrayType(mechanic_or_theme_schema, True),
        True,
    ),
    StructField(
        "themes",
        ArrayType(mechanic_or_theme_schema, True),
        True,
    ),
    StructField("theme_presence_score", IntegerType(), True),
    StructField("persistence", persistence_schema, True),
    StructField("perceived_persistence", perceived_persistence_schema, True),
    StructField("extraction_notes", ArrayType(StringType(), True), True),
])

# Parse only successful JSON responses from the current run.
structured_extractions_df = (
    spark.table(FINAL_PARSED_TABLE)
    .where(
        (F.col("run_id") == FINAL_RUN_ID)
        & (F.col("json_valid") == True)
    )
    .withColumn(
        "extraction",
        F.from_json(F.col("response_clean"), final_extraction_schema),
    )
)

# Expand the top-level JSON fields into columns. Evidence and repeated
# mechanics/themes remain nested arrays so their evidence stays attached.
expanded_extractions_df = structured_extractions_df.select(
    "game_name",
    "run_id",
    "saved_at",
    "parse_status",
    "extraction.*",
)

(
    expanded_extractions_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(FINAL_EXTRACTED_TABLE)
)

print(f"Saved structured extractions to: {FINAL_EXTRACTED_TABLE}")

display(
    spark.table(FINAL_EXTRACTED_TABLE)
    .where(F.col("run_id") == FINAL_RUN_ID)
    .select(
        "game_name",
        "game_format",
        "number_of_reels",
        "overall_minimum_bet",
        "overall_maximum_bet",
        "mechanics",
        "themes",
        "persistence",
        "perceived_persistence",
    )
    .orderBy("game_name")
)